# FINAL SCRIPT FOR SCRAPING,CLEANING, and WRITING TO BIGQUERY

In [90]:
#Import Required Libraries
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time
import json
import math
import random
import re
import numpy as np
import google.generativeai as genai
import ast
import warnings
warnings.filterwarnings("ignore")
from datetime import date
date_today = date.today()
import pandas_gbq
from google.cloud import bigquery
import os
import pyarrow as pa

#Import functions from other files
from job_scraper import *
from cleaning import *


# SCRAPE AND CLEAN

In [ ]:

### Scraping
url = ''
headers = {'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'}
pages_to_scrape = 1 #Place None if there is no limit

main_dict = {
    'job_ids' : [],
    'job_titles' : [],
    'companies' : [],
    'locations' : [],
    'disciplines' : [],
    'work_types' : [],
    'salaries' : [],
    'post_dates' : [],
    'job_details' : [],
    'links' : []
    }

scrape_to_csv_all(url, headers, main_dict, pages_to_scrape)

# Clean list data

# CLEANING

In [ ]:

df = pd.DataFrame(main_dict)

df = remove_duplicates(df)
df = df.reset_index()
df = df.drop(columns=['index'])
df = convert_datetime(df)
df = clean_job_details(df)
df = clean_disciplines(df)
#Cleaning salary
df = clean_salary(df)
#Extracting info from job descriptions
df = clean_descriptions(df)
#Cleaning Locations
df = clean_address(df)


# WRITE TO BIGQUERY

In [31]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'read-write-bq.json' #Credentials for Google Cloud Service Account
client =bigquery.Client()

In [96]:
# Convert to allowable datatypes in google bq
df.job_ids = df.job_ids.astype('int64')
df.day_posted = pd.to_datetime(df.day_posted)
df.time_posted = pd.to_datetime(df.time_posted, format="%H:%M:%S.%f")
df["responsibilities"] = df["responsibilities"].apply(lambda x: str(x))
df["tools"] = df["tools"].apply(lambda x: str(x))
df["education"] = df["education"].apply(lambda x: str(x))
df["yoe"] = df["yoe"].apply(lambda x: str(x))

In [40]:
project_id = '' #project id in your google cloud console
table_id = ''  #table id of your database


In [ ]:

job = client.load_table_from_dataframe(dataframe = df, destination=table_id)
job.result()